In [0]:
from pyspark.sql.functions import current_timestamp, col

# 1. (Feedback 2)
dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("schema", "janvander0912_bronze")
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("container", "janvander0912")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")

# 2. (Feedback 3)
base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
raw_data_path = f"{base_path}/raw_data/"

checkpoint_path = f"{base_path}/_checkpoints/fifa_v6/"
schema_path = f"{base_path}/_schemas/fifa_v6/"

table_name = f"{catalog}.{schema}.fifa_player_performance"


In [0]:
# 3. Autoloader
raw_stream_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path) 
    .option("header", "true")
    .option("inferSchema", "true")
    .load(raw_data_path)
)

In [0]:
# 4. Transformations 
bronze_stream_df = (raw_stream_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_filename")
    )
    .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
# Saving into bronze layer
sq = (bronze_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(table_name)
)

sq.awaitTermination()